# Exploración de Datos Landat 8/9 para CDMX

**Project:** Dos Méxicos Bajo el Mismo Sol

**Date:** 28 de Mayo de 2026

**Author:** Nelly Itzel Rodríguez Ortiz

In [2]:
# Import all libraries to use
import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

/home/nells-it/Documentos/PERSONAL/Portafolio/Project1-IslasCalor/project1.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [17]:
# Autenticación (abrirá un enlace en tu navegador)
ee.Authenticate()

# Inicializa con tu ID de proyecto de Google Cloud
proyecto_id = 'ee-nrodriguezo2301' 
ee.Initialize(project=proyecto_id)

# ¡Prueba que funciona!
print(ee.String('¡Conexión exitosa!').getInfo())

¡Conexión exitosa!


In [18]:
# Prueba rápida: crear un punto en CDMX y mostrarlo
cdmx_center = ee.Geometry.Point([-99.1332, 19.4326])
Map = geemap.Map(center=[19.4326, -99.1332], zoom=10)
Map.addLayer(cdmx_center, {'color': 'red'}, 'CDMX Centro')
Map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

In [19]:
# Colección: temperaturas superficiales listas para usar
landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')

print(f"📊 Número de imágenes en la colección: {landsat.size().getInfo()}")
print(f"📅 Rango de fechas: {landsat.first().date().getInfo()} a {landsat.sort('system:time_start', False).first().date().getInfo()}")

📊 Número de imágenes en la colección: 2014014
📅 Rango de fechas: {'type': 'Date', 'value': 1402322883992} a {'type': 'Date', 'value': 1779631180540}


In [20]:
# Definir AOI (Área de Interés) - CDMX + zona metropolitana
aoi = ee.Geometry.Rectangle([-99.4, 19.1, -98.9, 19.7])

# Filtrar colección
landsat_cdmx = landsat.filterBounds(aoi) \
                     .filterDate('2023-06-01', '2023-08-31') \
                     .filter(ee.Filter.lt('CLOUD_COVER', 20))

print(f"📊 Imágenes que cumplen criterios: {landsat_cdmx.size().getInfo()}")

# Ver las fechas disponibles
def print_dates(collection):
    dates = collection.aggregate_array('system:time_start').getInfo()
    for d in dates:
        print(f"  - {ee.Date(d).format('YYYY-MM-dd').getInfo()}")

print("📅 Fechas disponibles:")
print_dates(landsat_cdmx)

📊 Imágenes que cumplen criterios: 6
📅 Fechas disponibles:
  - 2023-06-01
  - 2023-06-17
  - 2023-07-19
  - 2023-06-01
  - 2023-06-17
  - 2023-07-19


In [21]:
# Ver bandas de la primera imagen
primera = landsat_cdmx.first()
bandas = primera.bandNames().getInfo()
print(f"🎵 Bandas disponibles ({len(bandas)}):")
for b in bandas:
    print(f"  - {b}")

🎵 Bandas disponibles (19):
  - SR_B1
  - SR_B2
  - SR_B3
  - SR_B4
  - SR_B5
  - SR_B6
  - SR_B7
  - SR_QA_AEROSOL
  - ST_B10
  - ST_ATRAN
  - ST_CDIST
  - ST_DRAD
  - ST_EMIS
  - ST_EMSD
  - ST_QA
  - ST_TRAD
  - ST_URAD
  - QA_PIXEL
  - QA_RADSAT


In [22]:
def kelvin_to_celsius(image):
    """Convierte la banda térmica ST_B10 de Kelvin a Celsius"""
    temp_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    return image.addBands(temp_celsius.rename('LST_Celsius'), None, True)

# Aplicar a la colección
landsat_temp = landsat_cdmx.map(kelvin_to_celsius)

print("✅ Banda LST_Celsius añadida a todas las imágenes")

✅ Banda LST_Celsius añadida a todas las imágenes


In [23]:
# Para evitar nubes, tomamos la MEDIANA de todas las imágenes del verano
# (la mediana es más robusta que la media, elimina outliers = nubes)
mosaico_verano = landsat_temp.median()

print("✅ Mosaico de verano 2023 creado (mediana de todas las imágenes)")

✅ Mosaico de verano 2023 creado (mediana de todas las imágenes)


In [24]:
# Crear mapa centrado en CDMX
Map = geemap.Map(center=[19.4326, -99.1332], zoom=11)

# Parámetros de visualización para temperatura en Celsius
vis_temp = {
    'min': 20,      # Temperatura mínima esperada
    'max': 50,      # Temperatura máxima esperada (isla de calor)
    'palette': ['blue', 'cyan', 'green', 'yellow', 'orange', 'red']
}

# Añadir capa de temperatura
Map.addLayer(mosaico_verano.select('LST_Celsius'), vis_temp, 'Temperatura Superficial (°C)')

# Añadir AOI
Map.addLayer(aoi, {'color': 'white'}, 'AOI CDMX')

# Añadir control de capas
Map.addLayerControl()

# Mostrar
Map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…